In [1]:
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import pandas as pd 
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
import openpyxl
df=pd.read_csv("BIOCBC.csv",encoding="cp1256",sep=";",usecols=[ 'سن بيمار',
       'جنسيت', 'محل درخواست','CBC,Diff-CBC & Diff-R.B.C',  'CBC,Diff-CBC & Diff-Hb',
         'CBC,Diff-CBC & Diff-Hct','CBC,Diff-CBC & Diff-W.B.C',"CBC,Diff-CBC & Diff-Platelet",
         "ESR 1st hr--ESR 1st hr","BS--Blood Sugar","FBS--F.B.S","BUN--B.U.N","Creatinine--Creatinine",
         "Choles--Cholesterol","TG--Triglyceride","LDL--LDL","HDL--HDL","SGOT--SGOT (AST)",
         "SGPT--SGPT(ALT)","Na--Serum Na","K--Serum K","HbA1c-Hb A1c-Hb A1c","CRP--CRP",
         "RF  به روش كمي--R.A. Factor"])
df=df.rename(columns={'سن بيمار':"age",'جنسيت':"sex",'CBC,Diff-CBC & Diff-R.B.C':"rbc",
 'محل درخواست':"ward",'CBC,Diff-CBC & Diff-Hb':"hb",'CBC,Diff-CBC & Diff-Hct':"hct",
 'CBC,Diff-CBC & Diff-W.B.C':"wbc","CBC,Diff-CBC & Diff-Platelet":"plt",
 "ESR 1st hr--ESR 1st hr":"esr","BS--Blood Sugar":"bs","FBS--F.B.S":"fbs","BUN--B.U.N":"bun",
         "Creatinine--Creatinine":"cr","Choles--Cholesterol":"chol","TG--Triglyceride":"tg",
         "LDL--LDL":"ldl","HDL--HDL":"hdl","SGOT--SGOT (AST)":"sgot","SGPT--SGPT(ALT)":"sgpt",
         "Na--Serum Na":"na","K--Serum K":"k","HbA1c-Hb A1c-Hb A1c":"a1c","CRP--CRP":"crp",
         "RF  به روش كمي--R.A. Factor":"rf"})
#pd.set_option('display.max_columns', None)
df["age"]=df["age"].str.replace("ساله","").str.replace("روزه","").str.replace(" ","")
df["age"]=pd.to_numeric(df["age"],errors="coerce")
df["ward"]=df["ward"].str.replace("بخش","").str.replace(" ","")
df["sex"]=df["sex"].str.replace("مرد","male").str.replace("زن","female")
#print(df.columns)
#print(df.head())
#print(df.shape)
#print(df.info())

diabet_cols=["fbs","sex","ward","age"]
lipid_cols=["ward","age","sex","tg","chol"]
kidney_cols=["ward","age","sex","cr","bun"]
liver_cols=["ward","age","sex","sgot","sgpt"]

print("diabet_col:",df[diabet_cols].notnull().all(axis=1).sum())
print("lipid_cols:",df[lipid_cols].notnull().all(axis=1).sum())
print("kidney_cols:",df[kidney_cols].notnull().all(axis=1).sum())
print("liver_cols:",df[liver_cols].notnull().all(axis=1).sum())

df_diabet=df[diabet_cols].dropna()
print(df_diabet.shape)
df_lipid=df[lipid_cols].dropna()
print(df_lipid.shape)
df_kidney=df[kidney_cols].dropna()
print(df_kidney.shape)
df_liver=df[liver_cols].dropna()
print(df_liver.shape)

df_diabet["fbs"]=df_diabet["fbs"].apply(lambda x: 0 if x<=120 else 1)
print(df_diabet["fbs"].value_counts())
df_lipid["tg"]=df_lipid["tg"].apply(lambda x: 0 if x<=160 else 1)
print(df_lipid["tg"].value_counts())
df_lipid["chol"]=df_lipid["chol"].apply(lambda x: 0 if x<=240 else 1)
print(df_lipid["chol"].value_counts())
df_kidney["cr"]=df_kidney["cr"].apply(lambda x: 0 if x<=1.5 else 1)
print(df_kidney["cr"].value_counts())
df_kidney["bun"]=df_kidney["bun"].apply(lambda x: 0 if x<=23 else 1)
print(df_kidney["bun"].value_counts())
df_liver["sgot"]=df_liver["sgot"].apply(lambda x: 0 if x<=40 else 1)
print(df_liver["sgot"].value_counts())
df_liver["sgpt"]=df_liver["sgpt"].apply(lambda x: 0 if x<40 else 1)
print(df_liver["sgpt"].value_counts())

df_kidney['sex'] = df_kidney['sex'].map({'male': 0, 'female': 1})
df_kidney["kidney"]=(df_kidney["cr"]==1)|(df_kidney["bun"]==1)
df_kidney["kidney"]=df_kidney["kidney"].astype(int)
ward_dummies=pd.get_dummies(df_kidney["ward"],prefix="ward")##
X_kidney=pd.concat([df_kidney[["age","sex",]],ward_dummies],axis=1)##
Y_kidney=df_kidney["kidney"]
X_train_k, X_test_k, Y_train_k, Y_test_k = train_test_split(X_kidney,
 Y_kidney, test_size=0.2, random_state=42)
tree_model=DecisionTreeClassifier(class_weight="balanced",random_state=42)
tree_model.fit(X_train_k,Y_train_k)
Y_pred_tree=tree_model.predict(X_test_k)
print("accuracy :",accuracy_score(Y_test_k,Y_pred_tree))
print(confusion_matrix(Y_test_k,Y_pred_tree))
print(classification_report(Y_test_k,Y_pred_tree))
scores=cross_val_score(tree_model,X_kidney,Y_kidney,cv=5,scoring="recall")######
print("Recall scores across folds:", scores)
print("Mean recall:", scores.mean())


X_kidney2=pd.concat([df_kidney[["age","sex",]],ward_dummies],axis=1)##
Y_kidney2=df_kidney["kidney"]
X_train_k2, X_test_k2, Y_train_k2, Y_test_k2 = train_test_split(X_kidney2,
 Y_kidney2, test_size=0.2, random_state=42)
tree_model2=DecisionTreeClassifier(class_weight="balanced",
max_depth=4,min_samples_leaf=5,random_state=42)#########
tree_model2.fit(X_train_k2,Y_train_k2)
Y_pred_tree2=tree_model2.predict(X_test_k2)
print("accuracy2 :",accuracy_score(Y_test_k2,Y_pred_tree2))
print(confusion_matrix(Y_test_k2,Y_pred_tree2))
print(classification_report(Y_test_k2,Y_pred_tree2))
cv_strategy=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)####
scores = cross_val_score(tree_model2, X_kidney2,Y_kidney2,cv=cv_strategy, scoring="recall")###
scores_p = cross_val_score(tree_model2, X_kidney2,Y_kidney2,cv=cv_strategy, scoring="precision")##
print("Recall scores across folds:", scores)
print("Mean recall:", scores.mean())
print("precision scores across folds:", scores_p)
print("Mean precision:", scores_p.mean())

X_kidney_f=pd.concat([df_kidney[["age","sex",]],ward_dummies],axis=1)##
Y_kidney_f=df_kidney["kidney"]
X_train_f, X_test_f, Y_train_f, Y_test_f = train_test_split(X_kidney_f,
 Y_kidney_f, test_size=0.2, random_state=42)
rf_model=RandomForestClassifier(class_weight="balanced",n_estimators=100,random_state=42)#####
rf_model.fit(X_train_f,Y_train_f)
Y_pred_rf=rf_model.predict(X_test_f)
print("accuracy_rf :",accuracy_score(Y_test_f,Y_pred_rf))
print(confusion_matrix(Y_test_f,Y_pred_rf))
print(classification_report(Y_test_f,Y_pred_rf))
scores = cross_val_score(rf_model, X_kidney_f, Y_kidney_f, cv=5, scoring='recall')######
print("Recall scores across folds:", scores)
print("Mean recall:", scores.mean())



diabet_col: 58
lipid_cols: 20
kidney_cols: 354
liver_cols: 95
(58, 4)
(20, 5)
(354, 5)
(95, 5)
fbs
0    40
1    18
Name: count, dtype: int64
tg
0    17
1     3
Name: count, dtype: int64
chol
0    19
1     1
Name: count, dtype: int64
cr
0    300
1     54
Name: count, dtype: int64
bun
0    295
1     59
Name: count, dtype: int64
sgot
0    68
1    27
Name: count, dtype: int64
sgpt
0    71
1    24
Name: count, dtype: int64
accuracy : 0.7464788732394366
[[50 13]
 [ 5  3]]
              precision    recall  f1-score   support

           0       0.91      0.79      0.85        63
           1       0.19      0.38      0.25         8

    accuracy                           0.75        71
   macro avg       0.55      0.58      0.55        71
weighted avg       0.83      0.75      0.78        71

Recall scores across folds: [0.3125 0.1875 0.25   0.3125 0.1875]
Mean recall: 0.25
accuracy2 : 0.647887323943662
[[42 21]
 [ 4  4]]
              precision    recall  f1-score   support

           0   

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=17d9599a-e07d-4c50-b1b1-f0833538d301' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>